## Sistema de Recomendação de Cartões de Crédito

Este notebook tem como objetivo:
1. Desenvolver um modelo preditivo para classificação do principal cartão para clientes
2. Aplicar o modelo de clientes na base prospects
3. Gerar arquivo final com cartão ideal (recomendação) para prospects

### Importar bibliotecas necessárias

In [ ]:
# EDA e Visualização de Dados
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, f_oneway
from colorama import Fore, Back, Style

# Configurar formato de exibição para não usar notação científica
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True, precision=5)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, log_loss
from catboost import CatBoostClassifier, Pool, cv

# Otimização
import optuna

# Utilitários
import joblib
import math

# Abrir Base Clientes

In [ ]:
df_clientes = pd.read_csv('datasets/clientes.csv')

In [ ]:
df_clientes.info()

In [ ]:
# Remover colunas únicas
df_clientes.drop(columns=['ID_Cliente', 'Nome'], axis=1, inplace=True)

In [ ]:
num_vars = df_clientes.select_dtypes(include=['number']).columns
cat_vars = df_clientes.select_dtypes(include=['object']).columns
target = 'Principal Cartão'

# EDA

## Testes de Hipóteses

In [ ]:
# Testes de hipóteses entre Target Categórica e Numéricas (ANOVA)
for num_col in num_vars:
    groups = [df_clientes[df_clientes[target] == val][num_col] for val in df_clientes[target].unique()]
    stat, p = f_oneway(*groups)
    print(f"{Fore.RED if p < 0.05 else Fore.WHITE}"
            f"ANOVA entre {num_col} e {target}: p-valor = {p}")

## Analisando relação entre variáveis explicativas e targets

In [ ]:

for col in num_vars:
    fig = px.box(df_clientes, x=target, y=col, title=f"{col} por {target}")
    fig.show()

for col in cat_vars:
    fig = px.histogram(df_clientes, x=col, color=target, barmode='group', title=f"{col} por {target}")
    fig.show()